In [1]:
%pip install torch transformers peft accelerate datasets bert-score pandas tqdm tabulate


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# -*- coding: utf-8 -*-
"""
정성 평가 스크립트 (CSV + 헤더 코멘트)
- 베이스/파인튜닝 답변 생성 → CSV 저장
- CSV 상단에 칼럼 설명을 주석 행으로 추가
- 재현성: do_sample=False, temperature=0.0, 고정 시드
- 동일 시스템 프롬프트 사용
"""

import os
import random
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# =========================
# 설정
# =========================
BASE_MODEL_PATH = "./kanana1_5_8b_instruct_2505"
ADAPTER_PATH    = "./kanana_finetuned_model02"

OUTPUT_CSV_PATH = "qualitative_comparison_kanana02.csv"   # 메인 산출물

QUESTIONS = [
    # 컨셉 기반 신규 디자인 생성
    "'한국의 전통적인 한옥과 수묵화'를 컨셉으로 제네시스 G90의 새로운 스페셜 에디션 모델을 디자인해줘. 특히 크레스트 그릴, 휠, 실내 내장재의 디자인이 어떻게 바뀔지 구체적으로 묘사해줘.",
    "2050년 미래 해양 도시를 탐험하기 위한 '현대 포세이돈'이라는 이름의 수륙양용 SUV를 상상해서 디자인해줘. 공기역학적인 차체, 잠수 모드를 위한 헤드라이트, 물 속 추진을 위한 휠의 변형 디자인을 중심으로.",
    # 특정 디자인 요소의 창의적 융합
    "현대자동차의 '파라메트릭 픽셀'과 제네시스의 '두 줄' 디자인을 융합해서, 새로운 전기 스포츠카의 테일램프 디자인을 만들어줘. 어떤 모양일지 아주 상세하게 설명해줘.",
    # 브랜드 아이덴티티 기반 신규 모델
    "현대의 고성능 'N' 브랜드에서 최초의 오프로드용 픽업트럭을 만든다면 어떤 모습일까? 'N' 브랜드의 상징색, 공격적인 범퍼 디자인, 그리고 거친 지형을 위한 타이어와 휠 디자인을 구체적으로 설명해줘."
]

SYSTEM_PROMPT = (
    "당신은 자동차 디자인 전문 AI입니다. "
    "질문에 대해 창의적이되 실현 가능한 디자인 컨셉을 제안하고, "
    "구체적 요소(그릴, 램프, 휠, 컬러/소재, 인테리어)를 명확히 기술하세요. "
    "한 문단 이상으로 충분히 상세히 작성하되, 불필요한 장황함은 피하고 논리적 일관성을 유지하세요."
)

# 재현성 확보
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

GEN_KW = dict(
    max_new_tokens=512,
    do_sample=False,   # 재현성
    temperature=0.0,   # 재현성
    top_p=1.0,
    repetition_penalty=1.05,
)

# =========================
# 유틸
# =========================
def load_model(model_path, adapter_path=None):
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path).merge_and_unload()
    return model, tokenizer

@torch.no_grad()
def generate_answer(model, tokenizer, question):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    # 문자열 프롬프트
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        eos_token_id=tokenizer.eos_token_id,
        **GEN_KW
    )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 안전 파싱: 입력(prompt) 길이만큼 잘라서 남은 부분을 답변으로 간주
    prompt_len_str = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
    answer = decoded[len(prompt_len_str):].strip()
    return answer

# =========================
# 메인
# =========================
def main():
    # 모델 로드
    base_model, tok = load_model(BASE_MODEL_PATH)
    ft_model, _     = load_model(BASE_MODEL_PATH, ADAPTER_PATH)

    # 생성
    base_answers, ft_answers = [], []
    print("\n--- 베이스 모델 생성 ---")
    for q in tqdm(QUESTIONS, desc="Base"):
        base_answers.append(generate_answer(base_model, tok, q))

    print("\n--- 파인튜닝 모델 생성 ---")
    for q in tqdm(QUESTIONS, desc="Finetuned"):
        ft_answers.append(generate_answer(ft_model, tok, q))

    # 데이터프레임 (채점 칼럼 포함)
    df = pd.DataFrame({
        "Question": QUESTIONS,
        "Base_Answer": base_answers,
        "Finetuned_Answer": ft_answers,
        "creativity": "",
        "specificity": "",
        "coherence": "",
        "brand_fit": "",
        "overall": "",
        "rater": "",
        "notes": "",
    })

    # CSV 저장 (맨 위에 컬럼별 설명 주석 행 추가)
    with open(OUTPUT_CSV_PATH, "w", encoding="utf-8-sig") as f:
        f.write("# Question: 평가 질문\n")
        f.write("# Base_Answer: 베이스 모델 답변\n")
        f.write("# Finetuned_Answer: 파인튜닝 모델 답변\n")
        f.write("# creativity: 창의성 (1~5)\n")
        f.write("# specificity: 구체성 (1~5)\n")
        f.write("# coherence: 일관성 (1~5)\n")
        f.write("# brand_fit: 브랜드 적합성 (1~5)\n")
        f.write("# overall: 종합 점수 (1~5)\n")
        f.write("# rater: 평가자 이름/ID\n")
        f.write("# notes: 평가자 자유 코멘트\n")
        df.to_csv(f, index=False)

    print(f"\n✅ CSV 저장 완료: {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    main()


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


--- 베이스 모델 생성 ---


Base: 100%|██████████| 4/4 [01:04<00:00, 16.03s/it]



--- 파인튜닝 모델 생성 ---


Finetuned: 100%|██████████| 4/4 [00:47<00:00, 11.94s/it]


✅ CSV 저장 완료: qualitative_comparison_kanana02.csv


In [3]:
# -*- coding: utf-8 -*-
"""
정량 평가 스크립트 (CSV 전용, 지표 = Precision/Recall/F1_Score/CtxAcc)
- 모델 출력 JSON에서 answer / context number 분리
- BERTScore(Precision/Recall/F1_Score)는 answer만으로 계산
- CtxAcc(문맥 정확도)는 context number 정확도
- 결과는 CSV 한 파일로 저장
"""

import re
import json
import unicodedata
from typing import List, Dict, Any, Tuple

import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import bert_score

# -----------------------------
# 경로 설정
# -----------------------------
BASE_MODEL_PATH = "./kanana1_5_8b_instruct_2505"
ADAPTER_PATH    = "./kanana_finetuned_model02"
TEST_DATA_PATH  = "./test02.jsonl"
OUTPUT_CSV_PATH = "./quantitative_evaluation_kanana02.csv"

# 훈련 시 시스템 프롬프트와 동일
SYSTEM_PROMPT = (
    "당신은 자동차 디자인 트렌드와 역사에 정통한 '자동차 디자인 전문 AI'입니다. "
    "특히 현대자동차의 디자인 철학인 '센슈어스 스포티니스'와 '플루이딕 스컬프처'를 깊이 이해하고 있습니다. "
    "사용자의 질문에 대해, 제공된 문맥을 참고하여 전문 지식을 바탕으로 상세하게 설명해주세요. "
    "답변은 반드시 아래 예시와 같이 JSON 형식으로 생성해야 하며, 어떤 문맥을 참고했는지 `context number` 필드에 해당 인덱스를 '[숫자]' 형식으로 포함해야 합니다."
    "\n\n"
    "답변 예시: {\"context number\": \"[1]\", \"answer\": \"플루이딕 스컬프처는 물이나 바람이 흐르는 듯한 유기적인 선을 강조하는 디자인 철학입니다.\"}"
)

# -----------------------------
# 유틸: JSON 안전 파싱 / 정규화 / context index
# -----------------------------
JSON_OBJ_RE = re.compile(r"\{.*?\}", flags=re.S)
CTX_NUM_RE  = re.compile(r'\[\s*(\d+)\s*\]')

def safe_json_answer(text: str) -> Tuple[str, str]:
    """
    모델 출력에서 마지막 JSON 객체를 찾아 answer와 context number를 분리 추출.
    실패 시: answer는 전체 텍스트, context는 텍스트에서 [i] 패턴 탐색.
    """
    answer = text.strip()
    ctx_raw = ""
    cands = JSON_OBJ_RE.findall(text)
    for s in reversed(cands):
        try:
            obj = json.loads(s)
            if isinstance(obj, dict):
                if "answer" in obj:
                    answer = str(obj["answer"]).strip()
                if "context number" in obj:
                    ctx_raw = str(obj["context number"]).strip()
                return answer, ctx_raw
        except Exception:
            continue
    m = CTX_NUM_RE.search(text)
    if m:
        ctx_raw = f"[{m.group(1)}]"
    return answer, ctx_raw

def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFKC", str(s)).strip()
    return re.sub(r"\s+", " ", s)

def to_ctx_index(ctx_raw: str) -> int:
    m = CTX_NUM_RE.search(ctx_raw)
    return int(m.group(1)) if m else 0

# -----------------------------
# 데이터 로드/포맷
# -----------------------------
def load_test(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def format_data_for_eval(raw_data: List[Dict[str, Any]]):
    """
    positive_index의 단일 문맥만 프롬프트에 포함.
    contexts 항목은 이미 "[i] ..." 형태이므로 prefix를 추가하지 않음.
    """
    out = []
    for it in raw_data:
        q   = it.get("question")
        a   = it.get("answer")
        ctxs = it.get("contexts") or []
        pix  = it.get("positive_index")
        if not q or not a or not ctxs or pix is None:
            continue
        positive_context = ctxs[pix - 1]  # 1-based index
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "system", "content": f"다음은 참고 문맥입니다:\n{positive_context}"},
            {"role": "user",   "content": q},
        ]
        out.append({
            "question": q,
            "reference_answer": a,
            "reference_ctx_number": f"[{pix}]",
            "messages": messages
        })
    return out

# -----------------------------
# 생성 & 평가
# -----------------------------
@torch.no_grad()
def generate_answer(model, tokenizer, messages) -> str:
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def evaluate_model(name: str, model, tok, data) -> pd.DataFrame:
    preds_ans, preds_ctx, refs_ans, refs_ctx, raws, qs = [], [], [], [], [], []
    for item in tqdm(data, desc=f"{name} 평가 중"):
        out_text = generate_answer(model, tok, item["messages"])
        ans_text, ctx_raw = safe_json_answer(out_text)
        preds_ans.append(ans_text)
        preds_ctx.append(ctx_raw)
        refs_ans.append(item["reference_answer"])
        refs_ctx.append(item["reference_ctx_number"])
        raws.append(out_text)
        qs.append(item["question"])

    # BERTScore (answer만)
    bert_p, bert_r, bert_f1 = bert_score.score(
        preds_ans, refs_ans, lang="ko",
        model_type="bert-base-multilingual-cased", verbose=False
    )

    # CtxAcc (context number 정확도)
    ctx_acc = [to_ctx_index(p) == to_ctx_index(r) for p, r in zip(preds_ctx, refs_ctx)]

    df = pd.DataFrame({
        "Question": qs,
        "Reference Answer": refs_ans,
        "Reference_CtxNumber": refs_ctx,

        f"{name}_Answer": preds_ans,
        f"{name}_ContextNumber": preds_ctx,
        f"{name}_Answer_Raw": raws,

        f"{name}_Precision": bert_p.tolist(),
        f"{name}_Recall": bert_r.tolist(),
        f"{name}_F1_Score": bert_f1.tolist(),
        f"{name}_CtxAcc": ctx_acc,
    })
    return df

def main():
    # 데이터 준비
    raw_test  = load_test(TEST_DATA_PATH)
    test_data = format_data_for_eval(raw_test)
    print(f"✅ 테스트 샘플: {len(test_data)}")

    # 토크나이저 공통
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    # Base
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_PATH, trust_remote_code=True, device_map="auto", torch_dtype=torch.bfloat16
    )
    base_df = evaluate_model("Base", base_model, tok, test_data)
    del base_model; torch.cuda.empty_cache()

    # Finetuned (LoRA merge)
    base_for_ft = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_PATH, trust_remote_code=True, device_map="auto", torch_dtype=torch.bfloat16
    )
    ft_model = PeftModel.from_pretrained(base_for_ft, ADAPTER_PATH).merge_and_unload()
    ft_df = evaluate_model("Finetuned", ft_model, tok, test_data)
    del base_for_ft, ft_model; torch.cuda.empty_cache()

    # 병합
    df = pd.merge(
        base_df, ft_df,
        on=["Question", "Reference Answer", "Reference_CtxNumber"],
        suffixes=("_Base", "_Finetuned")
    )

    # CSV 저장 (전 행 전체)
    df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"\n✅ 결과 CSV 저장 완료: {OUTPUT_CSV_PATH}")

    # 콘솔 요약(평균)
    for metric in ["Precision", "Recall", "F1_Score", "CtxAcc"]:
        b = pd.to_numeric(df[f"Base_{metric}"], errors="coerce").mean()
        f = pd.to_numeric(df[f"Finetuned_{metric}"], errors="coerce").mean()
        print(f"{metric:10s} | Base {b:.4f} → Finetuned {f:.4f}")

if __name__ == "__main__":
    main()


✅ 테스트 샘플: 289


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Base 평가 중: 100%|██████████| 289/289 [18:21<00:00,  3.81s/it]


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Finetuned 평가 중: 100%|██████████| 289/289 [07:23<00:00,  1.53s/it]



✅ 결과 CSV 저장 완료: ./quantitative_evaluation_kanana02.csv
Precision  | Base 0.7611 → Finetuned 0.9130
Recall     | Base 0.8491 → Finetuned 0.9206
F1_Score   | Base 0.8011 → Finetuned 0.9161
CtxAcc     | Base 0.8097 → Finetuned 0.9965
